# ARI711S – Group Project | Part 2: Hospital Shift Scheduler
## Juliette's Section – Node Consistency & AC-3 Algorithm

---

### Overview
This notebook implements the constraint propagation phase of the 
Hospital Shift Scheduling CSP solver. My responsibility covers:

1. **`enforce_node_consistency()`** – removes nurses on leave (unary constraint)
2. **`revise(x, y)`** – arc consistency between two shift variables (binary constraint)
3. **`ac3()`** – propagates arc consistency across all relevant shift pairs

---

### Problem Formulation (CSP)

| CSP Element | Meaning |
|---|---|
| **Variables (X)** | 21 weekly shift slots (7 days × 3 shifts) |
| **Domain (D)** | Set of available nurses per shift |
| **Constraints (C)** | Unary: leave days · Binary: Night→Morning rest · Higher-Order: max 5 shifts |

### The Three Constraints
- **Unary**: A nurse cannot work on a day they have pre-approved leave.
- **Binary**: A nurse on a Night shift cannot work the Morning of the very next day.
- **Higher-Order**: No nurse works more than 5 shifts in the week *(handled by Rejoice)*.

In [20]:
from collections import deque

DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
SHIFTS = ["Morning", "Afternoon", "Night"]

ALL_VARIABLES = [f"{day}_{shift}" for day in DAYS for shift in SHIFTS]

print(f"Total shift variables: {len(ALL_VARIABLES)}")
for v in ALL_VARIABLES:
    print(f"  {v}")

Total shift variables: 21
  Monday_Morning
  Monday_Afternoon
  Monday_Night
  Tuesday_Morning
  Tuesday_Afternoon
  Tuesday_Night
  Wednesday_Morning
  Wednesday_Afternoon
  Wednesday_Night
  Thursday_Morning
  Thursday_Afternoon
  Thursday_Night
  Friday_Morning
  Friday_Afternoon
  Friday_Night
  Saturday_Morning
  Saturday_Afternoon
  Saturday_Night
  Sunday_Morning
  Sunday_Afternoon
  Sunday_Night


---
## Step 1 – Load Staff Data

We load nurse names and their leave days from a text file.
Each line has a nurse name followed by any days they are on leave.

In [21]:
def load_staff(filepath):
    """
    Load nurse data from a staff text file.
    Format: NurseName   LeaveDay1   LeaveDay2 ...
    Returns: nurses (list), leave (dict)
    """
    nurses = []
    leave = {}

    with open(filepath, "r") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parts = [p.strip() for p in line.replace(",", "\t").split("\t") if p.strip()]
            if not parts:
                continue
            name = parts[0]
            nurses.append(name)
            leave_days = {token for token in parts[1:] if token in DAYS}
            leave[name] = leave_days

    return nurses, leave

print("load_staff() defined.")

load_staff() defined.


---
## Step 2 – The Shift_AI_Solver Class

The solver class holds all domains and constraints.
My methods handle the preparation phase before backtracking:
- `enforce_node_consistency()` — unary constraint pruning
- `revise(x, y)` — binary arc consistency  
- `ac3()` — full AC-3 propagation

In [22]:
class Shift_AI_Solver:
    def __init__(self, nurses, leave):
        self.nurses = nurses
        self.leave = leave
        self.variables = ALL_VARIABLES[:]
        self.domains = {var: set(nurses) for var in self.variables}

    def enforce_node_consistency(self):
        print("Enforcing node consistency (unary constraints)...")
        for variable in self.variables:
            day = variable.split("_")[0]
            to_remove = {
                nurse for nurse in self.domains[variable]
                if day in self.leave.get(nurse, set())
            }
            self.domains[variable] -= to_remove
            if to_remove:
                print(f"  [{variable}] Removed (on leave): {to_remove}")
        print("Node consistency enforced.\n")

    def revise(self, x, y):
        x_day, x_type = x.rsplit("_", 1)
        y_day, y_type = y.rsplit("_", 1)

        if x_type != "Night" or y_type != "Morning":
            return False
        if DAYS.index(y_day) != DAYS.index(x_day) + 1:
            return False

        revised = False
        to_remove = set()

        for nurse in self.domains[x]:
            remaining_for_y = self.domains[y] - {nurse}
            if len(remaining_for_y) == 0:
                to_remove.add(nurse)
                revised = True

        self.domains[x] -= to_remove
        return revised

    def ac3(self):
        print("Running AC-3 algorithm...")
        queue = deque()
        for i in range(len(DAYS) - 1):
            night = f"{DAYS[i]}_Night"
            morning = f"{DAYS[i+1]}_Morning"
            queue.append((night, morning))

        print(f"  Initial arc queue: {len(queue)} arcs")

        while queue:
            x, y = queue.popleft()
            if self.revise(x, y):
                print(f"  Revised [{x}] → remaining domain: {self.domains[x]}")
                if len(self.domains[x]) == 0:
                    print(f"  FAILURE: [{x}] domain is empty!")
                    return False
                x_day, x_type = x.rsplit("_", 1)
                if x_type == "Night":
                    x_idx = DAYS.index(x_day)
                    if x_idx > 0:
                        prev_night = f"{DAYS[x_idx - 1]}_Night"
                        queue.append((prev_night, x))

        print("AC-3 complete. All arcs are consistent.\n")
        return True

    def print_domains(self):
        print("\nDomain Summary:")
        for var in self.variables:
            nurses_list = sorted(self.domains[var])
            print(f"  {var:<25} ({len(nurses_list)} nurses): {nurses_list}")

print("Shift_AI_Solver class defined successfully.")

Shift_AI_Solver class defined successfully.


---
## Step 3 – Create Test Data & Run

We create a sample `staff_small.txt` file and run the full pipeline.

In [23]:
sample_staff = """# staff_small.txt
Kamati R\tMonday
Swartbooi I\tTuesday
Mudge D
Naruseb J\tWednesday
Eigowab S\tFriday
Beukes D\tSaturday
Kahuure K
Hanse-Himarwa K\tThursday
Mwandingi F\tSunday
Iivula-Ithana P
Gaweseb T\tFriday
Kambonde A\tMonday\tWednesday
"""

with open("staff_small.txt", "w") as f:
    f.write(sample_staff)

print("staff_small.txt created.")

staff_small.txt created.


In [24]:
nurses, leave = load_staff("staff_small.txt")

print(f"Loaded {len(nurses)} nurses:\n")
for name in nurses:
    leave_days = leave[name] if leave[name] else {"None"}
    print(f"  {name:<22} Leave: {leave_days}")

Loaded 12 nurses:

  Kamati R               Leave: {'Monday'}
  Swartbooi I            Leave: {'Tuesday'}
  Mudge D                Leave: {'None'}
  Naruseb J              Leave: {'Wednesday'}
  Eigowab S              Leave: {'Friday'}
  Beukes D               Leave: {'Saturday'}
  Kahuure K              Leave: {'None'}
  Hanse-Himarwa K        Leave: {'Thursday'}
  Mwandingi F            Leave: {'Sunday'}
  Iivula-Ithana P        Leave: {'None'}
  Gaweseb T              Leave: {'Friday'}
  Kambonde A             Leave: {'Wednesday', 'Monday'}


### 3a – enforce_node_consistency()

Applies the **unary constraint**: nurses on leave are removed from 
all shift domains for that day.

In [25]:
solver = Shift_AI_Solver(nurses, leave)

print("BEFORE node consistency:")
print(f"  Friday_Morning domain size: {len(solver.domains['Friday_Morning'])}")
print(f"  Monday_Night domain size:   {len(solver.domains['Monday_Night'])}")
print()

solver.enforce_node_consistency()

print("AFTER node consistency:")
print(f"  Friday_Morning: {sorted(solver.domains['Friday_Morning'])}")
print(f"  Monday_Night:   {sorted(solver.domains['Monday_Night'])}")

BEFORE node consistency:
  Friday_Morning domain size: 12
  Monday_Night domain size:   12

Enforcing node consistency (unary constraints)...
  [Monday_Morning] Removed (on leave): {'Kamati R', 'Kambonde A'}
  [Monday_Afternoon] Removed (on leave): {'Kamati R', 'Kambonde A'}
  [Monday_Night] Removed (on leave): {'Kamati R', 'Kambonde A'}
  [Tuesday_Morning] Removed (on leave): {'Swartbooi I'}
  [Tuesday_Afternoon] Removed (on leave): {'Swartbooi I'}
  [Tuesday_Night] Removed (on leave): {'Swartbooi I'}
  [Wednesday_Morning] Removed (on leave): {'Naruseb J', 'Kambonde A'}
  [Wednesday_Afternoon] Removed (on leave): {'Naruseb J', 'Kambonde A'}
  [Wednesday_Night] Removed (on leave): {'Naruseb J', 'Kambonde A'}
  [Thursday_Morning] Removed (on leave): {'Hanse-Himarwa K'}
  [Thursday_Afternoon] Removed (on leave): {'Hanse-Himarwa K'}
  [Thursday_Night] Removed (on leave): {'Hanse-Himarwa K'}
  [Friday_Morning] Removed (on leave): {'Gaweseb T', 'Eigowab S'}
  [Friday_Afternoon] Removed (on 

### 3b – ac3()

Applies the **binary constraint** (Night → next Morning rest rule) 
through AC-3 propagation across all 6 Night→Morning arc pairs.

In [26]:
success = solver.ac3()

if success:
    print("✅ AC-3 succeeded. Domains are arc-consistent.")
else:
    print("❌ AC-3 failed. No valid schedule possible.")

Running AC-3 algorithm...
  Initial arc queue: 6 arcs
AC-3 complete. All arcs are consistent.

✅ AC-3 succeeded. Domains are arc-consistent.


In [27]:
solver.print_domains()


Domain Summary:
  Monday_Morning            (10 nurses): ['Beukes D', 'Eigowab S', 'Gaweseb T', 'Hanse-Himarwa K', 'Iivula-Ithana P', 'Kahuure K', 'Mudge D', 'Mwandingi F', 'Naruseb J', 'Swartbooi I']
  Monday_Afternoon          (10 nurses): ['Beukes D', 'Eigowab S', 'Gaweseb T', 'Hanse-Himarwa K', 'Iivula-Ithana P', 'Kahuure K', 'Mudge D', 'Mwandingi F', 'Naruseb J', 'Swartbooi I']
  Monday_Night              (10 nurses): ['Beukes D', 'Eigowab S', 'Gaweseb T', 'Hanse-Himarwa K', 'Iivula-Ithana P', 'Kahuure K', 'Mudge D', 'Mwandingi F', 'Naruseb J', 'Swartbooi I']
  Tuesday_Morning           (11 nurses): ['Beukes D', 'Eigowab S', 'Gaweseb T', 'Hanse-Himarwa K', 'Iivula-Ithana P', 'Kahuure K', 'Kamati R', 'Kambonde A', 'Mudge D', 'Mwandingi F', 'Naruseb J']
  Tuesday_Afternoon         (11 nurses): ['Beukes D', 'Eigowab S', 'Gaweseb T', 'Hanse-Himarwa K', 'Iivula-Ithana P', 'Kahuure K', 'Kamati R', 'Kambonde A', 'Mudge D', 'Mwandingi F', 'Naruseb J']
  Tuesday_Night             (11 nurs

---
## Step 4 – Unit Tests for revise()

We test the binary arc consistency logic in isolation.

In [28]:
print("=== Test 1: Nurse removed when sole option for next Morning ===")
t = Shift_AI_Solver(["Nurse_A", "Nurse_B"], {})
t.domains["Tuesday_Morning"] = {"Nurse_A"}
t.domains["Monday_Night"] = {"Nurse_A", "Nurse_B"}
changed = t.revise("Monday_Night", "Tuesday_Morning")
assert changed == True
assert "Nurse_A" not in t.domains["Monday_Night"]
print(f"  Monday_Night after revise: {t.domains['Monday_Night']}")
print("  ✅ PASSED\n")

print("=== Test 2: No change for non-binary-constraint pairs ===")
t2 = Shift_AI_Solver(["Nurse_A", "Nurse_B"], {})
result = t2.revise("Monday_Morning", "Monday_Afternoon")
assert result == False
print(f"  Result: {result} — correctly False")
print("  ✅ PASSED\n")

print("=== Test 3: No change for non-consecutive days ===")
t3 = Shift_AI_Solver(["Nurse_A", "Nurse_B"], {})
result = t3.revise("Monday_Night", "Wednesday_Morning")
assert result == False
print(f"  Result: {result} — correctly False")
print("  ✅ PASSED")

=== Test 1: Nurse removed when sole option for next Morning ===
  Monday_Night after revise: {'Nurse_B'}
  ✅ PASSED

=== Test 2: No change for non-binary-constraint pairs ===
  Result: False — correctly False
  ✅ PASSED

=== Test 3: No change for non-consecutive days ===
  Result: False — correctly False
  ✅ PASSED


---
## Summary

| Method | Constraint Type | Effect |
|---|---|---|
| `enforce_node_consistency()` | Unary | Removed nurses on leave from shift domains |
| `revise(x, y)` | Binary | Removed nurses from Night domains where they were the sole option for next Morning |
| `ac3()` | Binary (propagated) | Applied revise() across all 6 Night→Morning arcs with ripple-effect re-checking |

After both steps, all 21 shift domains are pre-pruned and arc-consistent.
Rejoice's backtracking can now assign nurses efficiently without 
violating unary or binary constraints.